In [1]:
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
import sys
from chemrixs.smalldata import SmallData
%matplotlib widget 

In [2]:
base_folder =  '/sdf/data/lcls/ds/rix/'
run = 76

exp = 'rix100836924'
#79,81,82,83,85,94,95,96,98



fname = f'/sdf/data/lcls/ds/rix/rix100836924/scratch/h5/rix100836924_Run{run:04d}.h5'
#fname = f'{base_folder}/{exp}/hdf5/smalldata/{exp}_Run{run:04d}.h5'
with h5py.File(fname, 'r') as fh:
    data = SmallData(fh)

In [3]:
data.singleshot.process()

accessing SSdat
initialising <HDF5 group "/c_piranha" (1 members)>
setting up properties
initialising <HDF5 group "/det_crix_w8" (2 members)>
setting up properties
initialising <HDF5 group "/det_rix_fim0" (2 members)>
setting up properties
initialising <HDF5 group "/det_rix_fim1" (2 members)>
setting up properties
initialising <HDF5 group "/lightStatus" (2 members)>
setting up properties
initialising <HDF5 group "/mono_hrencoder" (1 members)>
setting up properties
initialising <HDF5 group "/timing" (5 members)>
setting up properties


In [4]:
data.singleshot.fim0

dask.array<add, shape=(6125937,), dtype=float64, chunksize=(10000,), chunktype=numpy.ndarray>

In [7]:
def sum_fims(fim_wfs,fim_roi,fim_bg_roi):
    '''Subtract background and integrate an ROI on the fims
    fim_roi [] === integration ROI
    '''    
    fim_proc =subtract_bg(fim_wfs,fim_bg_roi,invert = True)
    fim_wf_avs = np.nanmean(fim_proc,axis = 0)
    # fim = np.sum(np.abs(fim_proc[:,:,fim_roi[0]:fim_roi[1]]),axis = (1,2))

    fim = np.nansum(fim_proc[:,:,fim_roi[0]:fim_roi[1]],axis = (1,2))

    fim_sum_all = np.nansum(np.abs(fim_proc[:,:,fim_roi[0]:fim_roi[1]]),axis = 2)
    return fim, fim_wf_avs, fim_sum_all


def subtract_bg(wf,bg_roi=[0,50]):
    bg = np.mean(wf[...,bg_roi[0]:bg_roi[1]],axis= -1)
    shape = wf.shape    
    return wf - bg[...,np.newaxis]

In [ ]:
data.singleshot.fim_0.full_area
bg_roi=[0,50]
bg = np.mean(data.singleshot.fim_0.full_area[...,bg_roi[0]:bg_roi[1]],axis= -1)
bgf = data.singleshot.fim_0.full_area - bg[...,np.newaxis]

In [11]:
fim_roi=[60,70]

fim_proc =subtract_bg(data.singleshot.fim_0.full_area,[0,50])
fim_wf_avs = np.nanmean(fim_proc,axis = 0)
# fim = np.sum(np.abs(fim_proc[:,:,fim_roi[0]:fim_roi[1]]),axis = (1,2))

# fim = np.nansum(fim_proc[:,:,fim_roi[0]:fim_roi[1]],axis = (1,2))

fim_sum_all = np.nansum(np.abs(fim_proc[:,:,fim_roi[0]:fim_roi[1]]),axis = 2)

In [16]:
fim_sum_all[:100,0].compute()

array([   0.  ,  136.  ,  542.84,  708.04,  759.  ,  452.76,  691.84,
       1502.2 ,  940.64,  476.72,  300.4 ,  402.48,  463.52, 1012.  ,
        400.2 ,  610.92,  416.  ,  264.6 , 1145.96, 1432.92, 6227.52,
        706.92,  460.16,  393.52,  463.96,  471.88,  251.56,  563.16,
        847.16,  354.8 ,  461.  ,  260.76,  412.28,  858.4 , 2116.68,
       1385.  ,  631.6 ,  645.16,  264.64,  809.4 ,  502.08,  708.96,
        976.12,  401.  ,  494.2 ,  455.6 ,  469.  , 1094.24,  827.28,
       1162.36,  649.8 ,  313.2 ,  470.2 ,  361.36,  480.32,  264.64,
        442.72,  811.  ,  742.84,  723.  ,  432.  ,  581.32,  620.04,
        902.6 ,  528.68,  388.32,  833.84,  631.36,  948.56,  514.6 ,
       1070.  ,  940.84,  334.84,  313.44,  645.  ,  944.  ,  678.  ,
       1190.92,  944.24,  471.  ,  315.28,  796.56, 1029.32,  186.36,
        209.28,  492.44,  313.08,  818.08,  521.88,  902.68,  676.2 ,
       1249.16, 1090.96, 1383.64,  677.12,  233.64,  822.8 ,  580.8 ,
        263.32,  533

In [ ]:
andor_dir_dict = {
    'count': 'count',
    'full_area': 'full_area',
    'eventcodes': 'timing_sum_eventcodes',
    'apds': 'det_crix_w8_sum_full_area',
    'fim_0': 'det_rix_fim0_sum_full_area',
    'fim_1': 'det_rix_fim1_sum_full_area',
    'mono_encoder': 'mono_hrencoder_sum_value',
    'piranha': 'c_piranha_sum_full_area'}
andor_vls_dict = andor_dir_dict # assuming both detectors have the same keys

axis_svls_dict = andor_dir_dict # ass

detectors = {'andor_dir': andor_dir_dict,
        'andor_vls': andor_vls_dict,
        'axis_svls': axis_svls_dict,
        }

In [ ]:
detectors['andor_dir']